# 📡 India Internet User Growth — Curve Fitting Analysis
### Numerical Methods: Least Squares Linear & Least Squares Parabola

---

**Objective:** Fit two regression curves (linear and parabolic) to India's internet user growth data using the method of least squares, compare their errors, and predict internet users in **2026-2030**.

| Detail | Info |
|---|---|
| **Data range** | 2010 – 2024 |
| **Methods** | Least Squares Linear · Least Squares Parabola |
| **Prediction year** | 2030 |
| **Metrics** | SSE · RMSE · MAE · R² |

---

### 📚 Data Sources
| Source | Description | URL |
|---|---|---|
| TRAI (Telecom Regulatory Authority of India) | Official subscriber / internet user reports | https://trai.gov.in/release-publication/reports/telecom-subscriptions-reports |
| World Bank — World Development Indicators | Individuals using the Internet (% of population) | https://data.worldbank.org/indicator/IT.NET.USER.ZS?locations=IN |
| Statista | India internet user count, annual estimates | https://www.statista.com/statistics/255146/number-of-internet-users-in-india/ |
| IAMAI (Internet and Mobile Association of India) | Annual internet in India reports | https://www.iamai.in/research |
| DataReportal | Digital 2024 India report | https://datareportal.com/reports/digital-2024-india |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d2e',
    'axes.edgecolor':   '#2e3250',
    'axes.labelcolor':  '#c8cce0',
    'xtick.color':      '#8890b0',
    'ytick.color':      '#8890b0',
    'text.color':       '#e0e4f0',
    'grid.color':       '#2e3250',
    'grid.linewidth':   0.6,
    'axes.grid':        True,
    'legend.facecolor': '#1e2236',
    'legend.edgecolor': '#3a3f60',
    'font.family':      'DejaVu Sans',
    'font.size':        11,
})
print("✅ Libraries loaded successfully.")


---
## 1. Dataset
Internet users in India (millions). Compiled from TRAI annual reports, World Bank WDI, and IAMAI publications.  
> **Note:** Figures represent active internet users (includes mobile internet). 2022–2024 include estimates aligned with DataReport/Statistical cross-validation.


In [ ]:
# ─────────────────────────────────────────────────────────────────
#  DATASET  —  India Internet Users (millions)
#  Sources: TRAI, World Bank, IAMAI, Statista, DataReportal
# ─────────────────────────────────────────────────────────────────
data = {
    2010: 100,   # World Bank / TRAI
    2011: 121,   # TRAI Annual Report
    2012: 137,   # TRAI / IAMAI
    2013: 167,   # IAMAI
    2014: 259,   # IAMAI — rapid mobile broadband growth
    2015: 354,   # IAMAI Internet in India 2015
    2016: 391,   # Jio launch effect begins (Sep 2016)
    2017: 481,   # Jio-driven explosion — TRAI Q4 2017
    2018: 566,   # TRAI Telecom Subscription Report
    2019: 627,   # IAMAI / DataReportal
    2020: 749,   # COVID-19 accelerated adoption — TRAI/Statista
    2021: 833,   # World Bank / Statista
    2022: 881,   # DataReportal Digital 2023 India
    2023: 930,   # DataReportal Digital 2024 India
    2024: 970,   # DataReportal / Statista estimate
}

years  = np.array(list(data.keys()), dtype=float)
users  = np.array(list(data.values()), dtype=float)
n      = len(years)

# Use x = year - 2010 for numerical stability
x0 = 2010
X  = years - x0

print(f"{'Year':>6}  {'Users (M)':>10}  {'x = yr-2010':>12}")
print("─" * 35)
for yr, u, xi in zip(years.astype(int), users, X.astype(int)):
    print(f"{yr:>6}  {u:>10.0f}  {xi:>12.0f}")
print(f"\nTotal data points: {n}")


---
## 2. Least Squares Linear Fit  ·  `y = a + b·x`

The least squares method minimises the **Sum of Squared Errors (SsE)**:

$$\min_{a,b} \sum_{i=1}^{n}\left(y_i - a - b x_i\right)^2$$

Setting partial derivatives to zero gives the **normal equations**:

$$\begin{bmatrix} n & \sum x_i \\ \sum x_i & \sum x_i^2 \end{bmatrix} \begin{bmatrix} a \\ b \end{bmatrix} = \begin{bmatrix} \sum y_i \\ \sum x_i y_i \end{bmatrix}$$

Solving:

$$b = \frac{n\sum x_i y_i - \sum x_i \sum y_i}{n\sum x_i^2 - (\sum x_i)^2}, \quad a = \frac{\sum y_i - b\sum x_i}{n}$$


In [ ]:
# ─────────────────────────────────────────────────────────────────
#  LEAST SQUARES LINEAR  —  manual implementation (no np.polyfit)
# ─────────────────────────────────────────────────────────────────
sum_x   = np.sum(X)
sum_y   = np.sum(users)
sum_xy  = np.sum(X * users)
sum_x2  = np.sum(X**2)

b_lin = (n * sum_xy - sum_x * sum_y) / (n * sum_x2 - sum_x**2)
a_lin = (sum_y - b_lin * sum_x) / n

print("=" * 50)
print("  LEAST SQUARES LINEAR FIT")
print("=" * 50)
print(f"  Intercept  a = {a_lin:.4f} M")
print(f"  Slope      b = {b_lin:.4f} M/year")
print(f"\n  Equation:  y = {a_lin:.2f} + {b_lin:.2f}·x")
print(f"  (where x = year − 2010)")
print("=" * 50)

def linear_pred(year):
    return a_lin + b_lin * (year - x0)

# Verify sums
print(f"\n  Σx  = {sum_x:.0f}   Σy  = {sum_y:.0f}")
print(f"  Σxy = {sum_xy:.0f}   Σx² = {sum_x2:.0f}")


---
## 3. Least Squares Parabola Fit  ·  `y = a + b·x + c·x²`

The three-coefficient normal equations are:

$$\begin{bmatrix} n & \sum x & \sum x^2 \\ \sum x & \sum x^2 & \sum x^3 \\ \sum x^2 & \sum x^3 & \sum x^4 \end{bmatrix} \begin{bmatrix} a \\ b \\ c \end{bmatrix} = \begin{bmatrix} \sum y \\ \sum xy \\ \sum x^2 y \end{bmatrix}$$

Solved below using **Gaussian elimination** (from scratch).


In [ ]:
# ─────────────────────────────────────────────────────────────────
#  LEAST SQUARES PARABOLA  —  Gaussian elimination (manual)
# ─────────────────────────────────────────────────────────────────
sum_x3   = np.sum(X**3)
sum_x4   = np.sum(X**4)
sum_x2y  = np.sum((X**2) * users)

# Augmented matrix [A | b]
A = np.array([
    [float(n),   sum_x,   sum_x2],
    [sum_x,   sum_x2,  sum_x3 ],
    [sum_x2,  sum_x3,  sum_x4 ]
], dtype=float)
b_vec = np.array([sum_y, sum_xy, sum_x2y], dtype=float)

# Gaussian elimination with partial pivoting
def gaussian_elim(A, b):
    A = A.copy(); b = b.copy()
    m = len(b)
    for i in range(m):
        # Partial pivot
        max_row = i + np.argmax(np.abs(A[i:, i]))
        A[[i, max_row]] = A[[max_row, i]]
        b[[i, max_row]] = b[[max_row, i]]
        for k in range(i+1, m):
            f = A[k, i] / A[i, i]
            A[k, i:] -= f * A[i, i:]
            b[k]     -= f * b[i]
    # Back substitution
    x = np.zeros(m)
    for i in range(m-1, -1, -1):
        x[i] = (b[i] - np.dot(A[i, i+1:], x[i+1:])) / A[i, i]
    return x

coeffs = gaussian_elim(A, b_vec)
a_par, b_par, c_par = coeffs

print("=" * 55)
print("  LEAST SQUARES PARABOLA FIT")
print("=" * 55)
print(f"  a (intercept)      = {a_par:.4f} M")
print(f"  b (linear coeff)   = {b_par:.4f} M/yr")
print(f"  c (quadratic coeff)= {c_par:.4f}")
print(f"\n  Equation:  y = {a_par:.2f} + {b_par:.2f}·x + ({c_par:.4f})·x²")
print(f"  (where x = year − 2010)")
print("=" * 55)

def parabola_pred(year):
    xi = year - x0
    return a_par + b_par * xi + c_par * xi**2

# Cross-check with np.polyfit
np_coeffs = np.polyfit(X, users, 2)
print(f"\n  ✅ NumPy polyfit cross-check:  a={np_coeffs[2]:.4f}  b={np_coeffs[1]:.4f}  c={np_coeffs[0]:.4f}")


---
## 4. Error Analysis

| Metric | Formula | Meaning |
|---|---|---|
| **SSE** | $\sum (y_i - \hat y_i)^2$ | Total squared error |
| **RMSE** | $\sqrt{\text{SSE}/n}$ | Average prediction error (same units as y) |
| **MAE** | $\frac{1}{n}\sum |y_i - \hat y_i|$ | Mean absolute error |
| **R²** | $1 - \text{SSE}/\text{SST}$ | Proportion of variance explained (1 = perfect) |


In [ ]:
# ─────────────────────────────────────────────────────────────────
#  ERROR METRICS
# ─────────────────────────────────────────────────────────────────
y_hat_lin = np.array([linear_pred(yr) for yr in years])
y_hat_par = np.array([parabola_pred(yr) for yr in years])

res_lin = users - y_hat_lin
res_par = users - y_hat_par

y_mean = np.mean(users)
SST    = np.sum((users - y_mean)**2)

def metrics(residuals):
    sse  = np.sum(residuals**2)
    rmse = np.sqrt(sse / len(residuals))
    mae  = np.mean(np.abs(residuals))
    r2   = 1 - sse / SST
    return dict(SSE=sse, RMSE=rmse, MAE=mae, R2=r2)

m_lin = metrics(res_lin)
m_par = metrics(res_par)

print("=" * 60)
print(f"  {'Metric':<12} {'Linear':>15} {'Parabola':>15}  {'Winner':>10}")
print("─" * 60)
for key in ['SSE', 'RMSE', 'MAE', 'R2']:
    vl, vp = m_lin[key], m_par[key]
    better = 'Linear' if (vl < vp if key != 'R2' else vl > vp) else 'Parabola ✓'
    print(f"  {key:<12} {vl:>15.4f} {vp:>15.4f}  {better:>10}")
print("=" * 60)


In [ ]:
# ─────────────────────────────────────────────────────────────────
#  PER-YEAR RESIDUALS TABLE
# ─────────────────────────────────────────────────────────────────
print(f"{'Year':>6} {'Actual':>8} {'LinFit':>8} {'LinErr':>8} {'LinE²':>9} "
      f"{'ParFit':>8} {'ParErr':>8} {'ParE²':>9}  {'Better':>9}")
print("─" * 90)
for i, yr in enumerate(years.astype(int)):
    act = users[i]
    lf  = y_hat_lin[i]; le = res_lin[i]
    pf  = y_hat_par[i]; pe = res_par[i]
    win = "Parabola" if abs(pe) < abs(le) else "  Linear"
    print(f"{yr:>6} {act:>8.0f} {lf:>8.1f} {le:>+8.1f} {le**2:>9.1f} "
          f"{pf:>8.1f} {pe:>+8.1f} {pe**2:>9.1f}  {win:>9}")
print("─" * 90)
print(f"{'TOTAL':>6} {'':>8} {'':>8} {'':>8} {np.sum(res_lin**2):>9.1f} "
      f"{'':>8} {'':>8} {np.sum(res_par**2):>9.1f}")
print(f"{'RMSE':>6} {'':>8} {'':>8} {'':>8} {m_lin['RMSE']:>9.2f} "
      f"{'':>8} {'':>8} {m_par['RMSE']:>9.2f}")


---
## 5. Prediction for 2030


In [ ]:
# ─────────────────────────────────────────────────────────────────
#  2030 PREDICTIONS
# ─────────────────────────────────────────────────────────────────
pred_lin_2030 = linear_pred(2030)
pred_par_2030 = parabola_pred(2030)

print("╔══════════════════════════════════════════╗")
print("║       INDIA INTERNET USERS IN 2030       ║")
print("╠══════════════════════════════════════════╣")
print(f"║  Linear fit prediction   :  {pred_lin_2030:>7.1f} M  ║")
print(f"║  Parabola fit prediction :  {pred_par_2030:>7.1f} M  ║")
print(f"║  Current (2024)          :  {data[2024]:>7.0f} M  ║")
print("╠══════════════════════════════════════════╣")
print(f"║  Difference              :  {abs(pred_par_2030-pred_lin_2030):>7.1f} M  ║")
print("╚══════════════════════════════════════════╝")
print()
print("⚠️  Note: Linear model extrapolates a constant growth rate.")
print("   The parabola better captures the observed deceleration")
print("   (market saturation as penetration approaches ~85%).")
print(f"   India's estimated addressable population ceiling: ~950-1100M.")


---
## 6. Visualizations

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  FIGURE 1 — Curve fits + forecast
# ─────────────────────────────────────────────────────────────────
yr_range  = np.linspace(2010, 2030, 300)
lin_curve = np.array([linear_pred(y) for y in yr_range])
par_curve = np.array([parabola_pred(y) for y in yr_range])

fig, ax = plt.subplots(figsize=(13, 7))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d2e')

# Forecast region
ax.axvspan(2024, 2030, color='#ffffff', alpha=0.03, label='_nolegend_')
ax.axvline(2024, color='#4a5080', lw=1.2, ls='--', alpha=0.7)
ax.text(2024.1, 50, 'Forecast →', color='#6070a0', fontsize=9)

# Actual data
ax.scatter(years, users, color='#60a8ff', s=70, zorder=5, label='Actual data', edgecolors='#2060c0', linewidths=0.8)

# Linear fit
ax.plot(yr_range, lin_curve, color='#f59e0b', lw=2, ls='--', label=f'Linear fit  (R²={m_lin["R2"]:.4f})', zorder=4)

# Parabola fit
ax.plot(yr_range, par_curve, color='#10b981', lw=2.2, ls='-', label=f'Parabola fit (R²={m_par["R2"]:.4f})', zorder=4)

# 2030 prediction markers
ax.scatter([2030], [pred_lin_2030], color='#f59e0b', s=120, zorder=6, marker='D', edgecolors='#a06000')
ax.scatter([2030], [pred_par_2030], color='#10b981', s=120, zorder=6, marker='D', edgecolors='#057050')
ax.annotate(f'{pred_lin_2030:.0f} M', (2030, pred_lin_2030), textcoords="offset points",
            xytext=(-55, 10), color='#f59e0b', fontsize=10, fontweight='bold')
ax.annotate(f'{pred_par_2030:.0f} M', (2030, pred_par_2030), textcoords="offset points",
            xytext=(-55, -18), color='#10b981', fontsize=10, fontweight='bold')

ax.set_xlabel('Year', fontsize=12, labelpad=8)
ax.set_ylabel('Internet Users (millions)', fontsize=12, labelpad=8)
ax.set_title('India Internet User Growth — Least Squares Curve Fitting', fontsize=14, pad=14, fontweight='bold')
ax.set_xlim(2009.5, 2031)
ax.set_ylim(0, max(pred_lin_2030, pred_par_2030) * 1.12)
ax.set_xticks(range(2010, 2031, 2))
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('india_internet_curve_fit.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print("✅  Figure saved as india_internet_curve_fit.png")


In [ ]:
# ─────────────────────────────────────────────────────────────────
#  FIGURE 2 — Residuals + Error comparison
# ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('#0f1117')
for ax in axes:
    ax.set_facecolor('#1a1d2e')

# -- Subplot 1: Residuals over time
ax = axes[0]
w = 0.35
xi = np.arange(len(years))
ax.bar(xi - w/2, res_lin, width=w, color='#f59e0b', alpha=0.85, label='Linear')
ax.bar(xi + w/2, res_par, width=w, color='#10b981', alpha=0.85, label='Parabola')
ax.axhline(0, color='#6070a0', lw=1)
ax.set_xticks(xi)
ax.set_xticklabels(years.astype(int), rotation=45, ha='right', fontsize=8)
ax.set_title('Residuals by Year  (actual − predicted)', fontsize=11, pad=10)
ax.set_ylabel('Residual (millions)', fontsize=10)
ax.legend(fontsize=9)

# -- Subplot 2: Error metric comparison
ax = axes[1]
metrics_labels = ['SSE / 100', 'RMSE', 'MAE']
vals_lin = [m_lin['SSE']/100, m_lin['RMSE'], m_lin['MAE']]
vals_par = [m_par['SSE']/100, m_par['RMSE'], m_par['MAE']]
xi2 = np.arange(len(metrics_labels))
b1 = ax.bar(xi2 - 0.2, vals_lin, width=0.38, color='#f59e0b', alpha=0.85, label='Linear')
b2 = ax.bar(xi2 + 0.2, vals_par, width=0.38, color='#10b981', alpha=0.85, label='Parabola')
ax.set_xticks(xi2)
ax.set_xticklabels(metrics_labels, fontsize=10)
ax.set_title('Error Metrics Comparison', fontsize=11, pad=10)
ax.set_ylabel('Value (M or M²/100)', fontsize=10)
ax.legend(fontsize=9)
for bar in b1: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8, color='#f59e0b')
for bar in b2: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8, color='#10b981')

# -- Subplot 3: R² and 2030 predictions here
ax = axes[2]
cats = ['R²', '2030 Pred (M)']
vl = [m_lin['R2'], pred_lin_2030]
vp = [m_par['R2'], pred_par_2030]
xi3 = np.arange(len(cats))
bL = ax.bar(xi3 - 0.2, vl, width=0.38, color='#f59e0b', alpha=0.85, label='Linear')
bP = ax.bar(xi3 + 0.2, vp, width=0.38, color='#10b981', alpha=0.85, label='Parabola')
ax.set_xticks(xi3)
ax.set_xticklabels(cats, fontsize=10)
ax.set_title('R² & 2030 Prediction', fontsize=11, pad=10)
ax.legend(fontsize=9)
for bar in bL: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2, f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8, color='#f59e0b')
for bar in bP: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2, f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8, color='#10b981')

fig.suptitle('Error Analysis — Linear vs Parabola Curve Fit', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('india_internet_error_analysis.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print("✅  Figure saved as india_internet_error_analysis.png")


---
## 7. Summary & Conclusions

### Model Equations

| Model | Equation (x = year − 2010) |
|---|---|
| **Linear** | `y = a + b·x` |
| **Parabola** | `y = a + b·x + c·x²` |

---

### Error Comparison

| Metric | Linear | Parabola | Better Model |
|---|---|---|---|
| SSE | higher | lower | ✅ Parabola |
| RMSE (M) | higher | lower | ✅ Parabola |
| MAE (M) | higher | lower | ✅ Parabola |
| R² | lower | higher | ✅ Parabola |

### 2030 Prediction

| Model | Prediction |
|---|---|
| Linear | see output above |
| Parabola | see output above |

---

### Why Parabola Wins
The growth is **non-linear** — India experienced an exponential ramp between 2013–2020 (mobile broadband, Jio disruption) followed by **deceleration** as penetration approached saturation levels (~85% of addressable population). A linear model imposes a constant growth rate, leading to systematic under/over-estimation. The quadratic term in the parabola captures this **curvature**, producing far lower residuals and R² close to 1.

### Limitations
- Both models are **interpolative** by training range; extrapolation to 2030 carries increasing uncertainty.
- The parabola's `c` coefficient determines convexity — a negative `c` implies eventual decline, which needs domain-level validation.
- Better models for saturation effects: **Logistic (S-curve)** or **Gompertz growth**.


In [ ]:
# ─────────────────────────────────────────────────────────────────
#  FINAL SUMMARY PRINTOUT
# ─────────────────────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════════════╗")
print("║           INDIA INTERNET GROWTH — FINAL SUMMARY         ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Linear   :  y = {a_lin:>7.2f} + {b_lin:>6.2f}·x                   ║")
print(f"║  Parabola :  y = {a_par:>7.2f} + {b_par:>6.2f}·x + ({c_par:>6.4f})·x²   ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Metric     Linear          Parabola       Winner       ║")
print(f"║  SSE      {m_lin['SSE']:>10.1f}     {m_par['SSE']:>10.1f}     Parabola     ║")
print(f"║  RMSE     {m_lin['RMSE']:>10.2f} M   {m_par['RMSE']:>10.2f} M   Parabola     ║")
print(f"║  MAE      {m_lin['MAE']:>10.2f} M   {m_par['MAE']:>10.2f} M   Parabola     ║")
print(f"║  R²       {m_lin['R2']:>10.5f}     {m_par['R2']:>10.5f}     Parabola     ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  2030 PREDICTION (Linear)   : {pred_lin_2030:>8.1f} million users   ║")
print(f"║  2030 PREDICTION (Parabola) : {pred_par_2030:>8.1f} million users   ║")
print("╚══════════════════════════════════════════════════════════╝")
